# Day 1: TCGA-BRCA data loading, cleaning and overall survival

**Goal:** one clean table (patients x genes + survival) and a Kaplan-Meier curve for the whole cohort.

- `OS` = 1 means death, 0 means censored (still alive at last follow-up)
- `OS.time` = days from diagnosis to death or last follow-up
- TCGA barcode `TCGA-XX-XXXX-01A`: characters 14-15 are the sample type (`01` = primary tumor, `11` = normal)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from lifelines import KaplanMeierFitter

DATA = Path("../data")
FIGS = Path("../results/figures")
FIGS.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

## 1. Load the raw files

In [ ]:
# Expression: rows = genes (Ensembl IDs), columns = samples. Values are log2(TPM + 1).
expr = pd.read_csv(DATA / "TCGA-BRCA.star_tpm.tsv.gz", sep="\t", index_col=0).astype("float32")

# Survival: one row per sample, columns include OS, OS.time, _PATIENT
surv = pd.read_csv(DATA / "TCGA-BRCA.survival.tsv", sep="\t", index_col=0)

print("expression (genes x samples):", expr.shape)
print("survival   (samples x cols): ", surv.shape)
surv.head()

## 2. Inspect the survival table

In [ ]:
print(surv.dtypes, "\n")
print(surv[["OS", "OS.time"]].describe(), "\n")
print("Missing values:\n", surv[["OS", "OS.time"]].isna().sum())
print("\nEvents (deaths) vs censored:\n", surv["OS"].value_counts())

## 3. Clean the survival table
Keep primary tumors only, remove invalid follow-up times, and keep one sample per patient.

In [ ]:
log = {"start": len(surv)}

# 3a. primary tumor only (sample type code '01')
surv["sample_type"] = surv.index.str[13:15]
print(surv["sample_type"].value_counts(), "\n")
surv = surv[surv["sample_type"] == "01"]
log["primary tumor only"] = len(surv)

# 3b. drop missing / invalid follow-up
surv = surv.dropna(subset=["OS", "OS.time"])
surv = surv[surv["OS.time"] > 0]
log["valid OS/OS.time"] = len(surv)

# 3c. one sample per patient (keep first barcode alphabetically)
surv = surv.sort_index()
surv = surv[~surv["_PATIENT"].duplicated(keep="first")]
log["one sample per patient"] = len(surv)

surv["OS"] = surv["OS"].astype(int)
pd.Series(log, name="patients remaining")

## 4. Match expression and survival

In [ ]:
common = surv.index.intersection(expr.columns)
print(f"Samples with BOTH survival and expression: {len(common)}")

surv = surv.loc[common]
expr = expr[common]

# drop genes that are zero in every sample (no information)
expr = expr.loc[(expr > 0).any(axis=1)]
print("genes kept:", expr.shape[0])

## 5. Build the analysis table (samples x genes) and save it

In [ ]:
X = expr.T                      # rows = patients, columns = genes
X.index.name = "sample"
y = surv[["OS", "OS.time"]].copy()

assert X.index.equals(y.index), "Index mismatch between X and y"
assert X.isna().sum().sum() == 0, "NaNs in expression"

print("X:", X.shape, "| y:", y.shape)
print(f"Events: {y['OS'].sum()} deaths ({y['OS'].mean():.1%}), median follow-up: {y['OS.time'].median()/365.25:.1f} years")

DATA.joinpath("processed").mkdir(exist_ok=True)
X.to_pickle(DATA / "processed" / "X_expression.pkl")
y.to_pickle(DATA / "processed" / "y_survival.pkl")

## 6. Kaplan-Meier curve for the whole cohort

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(y["OS.time"] / 365.25, event_observed=y["OS"], label="All patients")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
kmf.plot_survival_function(ax=ax, ci_show=True, show_censors=True,
                           censor_styles={"ms": 4, "marker": "|"})
ax.set_xlabel("Time (years)")
ax.set_ylabel("Overall survival probability")
ax.set_title(f"TCGA-BRCA overall survival (n={len(y)})")
ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.savefig(FIGS / "km_overall.png", dpi=200)
plt.show()

print("Median survival (years):", kmf.median_survival_time_)
print("5-year survival:", float(kmf.predict(5)))

### Reading the plot
- The curve drops with each death; the small ticks are censored patients.
- Breast cancer survival is long, so the median is often not reached (`inf`). That is normal.
- A heavy tail beyond ~10 years has very few patients at risk, so be careful interpreting it.

**Next (Day 2):** clinical baseline with age + stage, Cox model, KM by stage.